In [ ]:
# نصب کتابخانه‌های موردنیاز
!pip install opencv-python-headless imutils moviepy

import cv2
import numpy as np
import os
from google.colab import drive
import matplotlib.pyplot as plt

# اتصال به Google Drive
drive.mount('/content/drive')

# ---------------------------
# بارگذاری دیتابیس تصاویر و ویدئوها
# ---------------------------
def load_image_db(folder):
    db = {}
    for f in os.listdir(folder):
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            name, _ = os.path.splitext(f)
            img = cv2.imread(os.path.join(folder, f))
            if img is not None:
                db[name] = img
                print(f"Loaded image: {name}")
    return db

def load_video_db(folder):
    db = {}
    for f in os.listdir(folder):
        if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
            name, _ = os.path.splitext(f)
            video_path = os.path.join(folder, f)
            db[name] = video_path
            print(f"Found trailer: {name}")
    return db

# ---------------------------
# پیدا کردن کتاب خاص در یک فریم
# ---------------------------
def find_specific_book_in_frame(frame, target_book_cover, detector, LOWE_RATIO=0.7, MIN_MATCHES=8):
    if frame is None or target_book_cover is None:
        return None

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frame_kp, frame_desc = detector.detectAndCompute(frame_gray, None)
    if frame_desc is None:
        return None

    cover_gray = cv2.cvtColor(target_book_cover, cv2.COLOR_BGR2GRAY)
    cover_kp, cover_desc = detector.detectAndCompute(cover_gray, None)
    if cover_desc is None:
        return None

    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)

    try:
        matches = flann.knnMatch(cover_desc, frame_desc, k=2)
        good_matches = []
        for match_pair in matches:
            if len(match_pair) == 2:
                m, n = match_pair
                if m.distance < LOWE_RATIO * n.distance:
                    good_matches.append(m)

        if len(good_matches) >= MIN_MATCHES:
            src_pts = np.float32([cover_kp[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
            dst_pts = np.float32([frame_kp[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

            H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
            if H is not None:
                h, w = target_book_cover.shape[:2]
                corners = np.float32([[0,0],[w,0],[w,h],[0,h]]).reshape(-1,1,2)
                transformed_corners = cv2.perspectiveTransform(corners, H)

                area = cv2.contourArea(transformed_corners)
                if area > 0.01 * (frame.shape[0] * frame.shape[1]):
                    return {
                        'homography': H,
                        'corners': transformed_corners,
                        'dst_corners': transformed_corners,
                        'matches': len(good_matches),
                        'inliers': np.sum(mask)
                    }
    except:
        pass

    return None

# ---------------------------
# جایگزین کردن جلد با فریم تریلر
# ---------------------------
def replace_cover_with_trailer_frame(frame, match_data, trailer_frame):
    if not match_data or trailer_frame is None:
        return frame

    H = match_data['homography']
    dst_corners = match_data['dst_corners']
    warped_trailer = cv2.warpPerspective(trailer_frame, H, (frame.shape[1], frame.shape[0]))

    mask_cover = np.zeros(frame.shape[:2], dtype=np.uint8)
    cv2.fillConvexPoly(mask_cover, np.int32(dst_corners), 255)

    inv_mask = cv2.bitwise_not(mask_cover)
    background = cv2.bitwise_and(frame, frame, mask=inv_mask)
    foreground = cv2.bitwise_and(warped_trailer, warped_trailer, mask=mask_cover)
    result = cv2.add(background, foreground)
    return result

# ---------------------------
# کلاس پردازش ویدئو
# ---------------------------
class OptimizedVideoProcessor:
    def __init__(self, book_db_path, trailer_db_path):
        self.detector = cv2.SIFT_create()
        print("Loading book covers database...")
        self.book_db = load_image_db(book_db_path)
        print(f"Loaded {len(self.book_db)} book covers")

        print("Loading trailers database...")
        self.trailer_db = load_video_db(trailer_db_path)
        print(f"Found {len(self.trailer_db)} trailers")

        self.main_book_title = None
        self.main_book_cover = None
        self.trailer_cap = None
        self.trailer_frame_count = 0
        self.trailer_total_frames = 0

    def get_next_trailer_frame(self):
        if not self.trailer_cap or not self.trailer_cap.isOpened():
            return None

        ret, trailer_frame = self.trailer_cap.read()
        if ret:
            self.trailer_frame_count += 1
            return trailer_frame
        else:
            self.trailer_cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            self.trailer_frame_count = 0
            ret, trailer_frame = self.trailer_cap.read()
            if ret:
                self.trailer_frame_count += 1
                return trailer_frame
        return None

    def process_video(self, input_video_path, output_video_path, max_frames=None, show_progress=True, target_book=None):
        if target_book is None:
            print("❌ Error: You must provide a target_book name.")
            return False

        if target_book not in self.book_db:
            print(f"❌ Target book '{target_book}' not found in book database.")
            return False

        self.main_book_title = target_book
        self.main_book_cover = self.book_db[target_book]
        print(f"🔎 Using target book: '{self.main_book_title}'")

        if self.main_book_title in self.trailer_db:
            self.trailer_cap = cv2.VideoCapture(self.trailer_db[self.main_book_title])
            self.trailer_total_frames = int(self.trailer_cap.get(cv2.CAP_PROP_FRAME_COUNT))
            self.trailer_frame_count = 0
            print(f"🎬 Trailer loaded: {self.trailer_total_frames} frames")
        else:
            print(f"⚠️ No trailer found for '{self.main_book_title}'. Replacement frames will be skipped.")

        cap = cv2.VideoCapture(input_video_path)
        if not cap.isOpened():
            print(f"Error: Cannot open video {input_video_path}")
            return False

        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if max_frames:
            total_frames = min(total_frames, max_frames)

        print(f"Video properties: {width}x{height}, {fps} FPS, {total_frames} frames")

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

        frame_count = 0
        detection_count = 0

        try:
            while True:
                ret, frame = cap.read()
                if not ret or (max_frames and frame_count >= max_frames):
                    break

                match_data = find_specific_book_in_frame(frame, self.main_book_cover, self.detector)

                if match_data:
                    detection_count += 1
                    if show_progress and frame_count % 30 == 0:
                        print(f"Frame {frame_count}: Found target book (matches: {match_data['matches']})")

                    trailer_frame = self.get_next_trailer_frame()
                    if trailer_frame is not None:
                        h, w = self.main_book_cover.shape[:2]
                        trailer_frame_resized = cv2.resize(trailer_frame, (w, h))
                        processed_frame = replace_cover_with_trailer_frame(frame, match_data, trailer_frame_resized)
                    else:
                        processed_frame = frame.copy()
                else:
                    processed_frame = frame.copy()
                    if show_progress and frame_count % 60 == 0:
                        print(f"Frame {frame_count}: Target book not visible")

                out.write(processed_frame)
                frame_count += 1

                if show_progress and frame_count % 30 == 0:
                    print(f"Processed {frame_count}/{total_frames} frames ({detection_count} detections)")

        except Exception as e:
            print(f"Error during processing: {e}")

        finally:
            cap.release()
            out.release()
            if self.trailer_cap:
                self.trailer_cap.release()

        detection_rate = (detection_count / frame_count) * 100 if frame_count > 0 else 0
        print(f"✅ Video processing completed!")
        print(f"   - Total frames processed: {frame_count}")
        print(f"   - Target book detections: {detection_count}")
        print(f"   - Detection rate: {detection_rate:.1f}%")
        print(f"   - Output saved to: {output_video_path}")

        return True

# ---------------------------
# اجرای تست
# ---------------------------
def process_test_video():
    book_db_path = '/content/drive/MyDrive/cover_db/covers'
    trailer_db_path = '/content/drive/MyDrive/cover_db/Trailers'
    input_video_path = '/content/drive/MyDrive/cover_db/Videos/The_Two_Towers.mp4'
    output_video_path = '/content/drive/MyDrive/cover_db/output_videos/processed_target_only.mp4'

    target_book_name = 'The_Lord_Of_The_Rings_Towers'

    os.makedirs(os.path.dirname(output_video_path), exist_ok=True)

    processor = OptimizedVideoProcessor(book_db_path, trailer_db_path)
    success = processor.process_video(
        input_video_path=input_video_path,
        output_video_path=output_video_path,
        max_frames=1000,
        show_progress=True,
        target_book=target_book_name
    )

    if success:
        print("video processing completed successfully!")

        cap = cv2.VideoCapture(output_video_path)
        sample_frames = []
        for i in range(0, 150, 30):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                sample_frames.append(frame)
        cap.release()

        if sample_frames:
            plt.figure(figsize=(15, 10))
            for i, frame in enumerate(sample_frames):
                plt.subplot(2, 3, i+1)
                plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                plt.title(f'Frame {i*30}')
                plt.axis('off')
            plt.tight_layout()
            plt.show()
    else:
        print(" video processing failed!")

print("🚀 Starting video processing...")
process_test_video()
